# Simulated Annealing for TSP

## Lab 4 – Work during the lab

**Tasks:**
1. Implement Simulated Annealing for TSP
2. Test on the same two instances as Lab 3: `pr107` (group A) and `zi929` (group B), with different parameter settings

**Instances:** `../tsp_instances/A/pr107.tsp` (107 cities), `../tsp_instances/B/zi929.tsp` (929 cities)

### Imports and global variables

In [1]:
import math
import os
import random
import time

### Helper functions

Reused from Lab 3: TSP file parser, distance matrix builder, tour cost, greedy initialisation, and O(1) 2-opt delta evaluation.

In [2]:
# ── TSP I/O ───────────────────────────────────────────────────────────────────

def load_tsp(file_name: str) -> list[tuple[float, float]]:
    """
    Parses a TSPLIB EUC_2D file.
    Returns a list of (x, y) city coordinates.
    """
    coords = []
    if not os.path.exists(file_name):
        print(f"Error: File '{file_name}' not found.")
        return coords

    in_node_section = False
    with open(file_name) as f:
        for line in f:
            line = line.strip()
            if line.startswith("NODE_COORD_SECTION"):
                in_node_section = True
                continue
            if line in ("EOF", ""):
                in_node_section = False
                continue
            if in_node_section:
                parts = line.split()
                if len(parts) >= 3:
                    coords.append((float(parts[1]), float(parts[2])))
    return coords


def build_distance_matrix(coords: list[tuple[float, float]]) -> list[list[float]]:
    """Pre-computes all pairwise Euclidean distances."""
    n = len(coords)
    dist = [[0.0] * n for _ in range(n)]
    for i in range(n):
        for j in range(i + 1, n):
            dx = coords[i][0] - coords[j][0]
            dy = coords[i][1] - coords[j][1]
            d = math.sqrt(dx * dx + dy * dy)
            dist[i][j] = d
            dist[j][i] = d
    return dist


def tour_cost(tour: list[int], dist: list[list[float]]) -> float:
    """Total cost of a closed TSP tour."""
    n = len(tour)
    return sum(dist[tour[i]][tour[(i + 1) % n]] for i in range(n))


# ── Greedy initialisation ─────────────────────────────────────────────────────

def greedy_tsp(dist: list[list[float]], start: int = 0) -> list[int]:
    """
    Nearest-neighbour greedy heuristic.
    Returns a tour (list of city indices).
    """
    n = len(dist)
    visited = [False] * n
    tour = [start]
    visited[start] = True
    for _ in range(n - 1):
        current = tour[-1]
        nearest = min(
            (j for j in range(n) if not visited[j]),
            key=lambda j: dist[current][j]
        )
        tour.append(nearest)
        visited[nearest] = True
    return tour


def verify_tour(tour: list[int], n: int) -> bool:
    """Checks that a tour is a valid permutation of all city indices."""
    return sorted(tour) == list(range(n))


# ── 2-opt neighbourhood ───────────────────────────────────────────────────────

def apply_2opt_move(tour: list[int], i: int, j: int) -> list[int]:
    """
    Applies a 2-opt move by reversing the segment tour[i+1 : j+1].
    Returns a new tour without modifying the original.
    """
    return tour[:i + 1] + tour[i + 1:j + 1][::-1] + tour[j + 1:]


def two_opt_delta(tour: list[int], dist: list[list[float]], i: int, j: int) -> float:
    """
    O(1) cost change of reversing tour[i+1 : j+1].
    Removes edges (tour[i]→tour[i+1]) and (tour[j]→tour[j+1 % n]).
    Adds    edges (tour[i]→tour[j])   and (tour[i+1]→tour[j+1 % n]).
    """
    n = len(tour)
    a, b = tour[i],     tour[i + 1]
    c, d = tour[j],     tour[(j + 1) % n]
    return dist[a][c] + dist[b][d] - dist[a][b] - dist[c][d]

### Core Algorithm

**Simulated Annealing (SA)** is a probabilistic local search metaheuristic inspired by the annealing process in metallurgy. Starting from a high temperature T, the algorithm progressively lowers T according to a *cooling schedule* g. At each step a random neighbour x is drawn; if x improves the current solution it is always accepted, otherwise it is accepted with probability e^((cost(c)−cost(x))/T). As T → 0 the acceptance of worsening moves becomes increasingly rare, focusing the search.

**Cooling schedules** (as per lab specification):
- **Geometric:** `T(k+1) = alpha * T(k)`, alpha ∈ (0,1) close to 1 — e.g. 0.999, 0.99, 0.95
- **Logarithmic:** `T(k+1) = T(k) / log(k+2)`
- **Linear:** `T(k+1) = T(k) / (k+1)`

**Neighbourhood:** a single random 2-opt move (reverse a random sub-segment of the tour).

**Pseudocode (from lab sheet):**
```
t = 0
initialise T
c = greedy_solution()          # warm start
evaluate c
best = c

repeat                         # outer loop: one temperature level per iteration
    repeat                     # inner loop: INNER_ITERS trials at current T
        x = random 2-opt neighbour of c
        if cost(x) < cost(c):
            c ← x
        else if random[0,1) < exp((cost(c) − cost(x)) / T):
            c ← x              # accept worsening move
        if cost(c) < cost(best):
            best ← c
    until (inner termination condition)
    T ← g(T, t)
    t ← t + 1
until (T < T_min)              # halting criterion

return best
```

**Key design choices:**
- **Warm start:** greedy nearest-neighbour tour from a random city (introduces run-to-run variance)
- **Neighbourhood:** single uniformly random 2-opt move per inner iteration — O(1) with delta eval
- **Halting:** outer loop stops when T < T_min

In [3]:
def simulated_annealing_tsp(
    dist: list[list[float]],
    initial_temp: float = 10000.0,
    min_temp: float = 0.1,
    alpha: float = 0.99,
    inner_iters: int = 100,
    cooling: str = "geometric",
) -> dict:
    """
    Simulated Annealing for TSP using 2-opt moves.

    Parameters:
        dist         : distance matrix (n x n)
        initial_temp : starting temperature T0
        min_temp     : halting criterion — stop when T < min_temp
        alpha        : cooling rate for geometric schedule (ignored otherwise)
        inner_iters  : number of random-move trials per temperature level
        cooling      : 'geometric' | 'log' | 'linear'

    Returns a dict with keys: 'tour', 'cost', 'iterations'
    """
    n = len(dist)

    # Warm start: greedy tour from a random city (gives run-to-run variance)
    start_city = random.randint(0, n - 1)
    current = greedy_tsp(dist, start=start_city)
    current_cost = tour_cost(current, dist)

    best = current[:]
    best_cost = current_cost

    T = initial_temp
    t = 0  # outer iteration counter (temperature level)

    while T > min_temp:
        # ── Inner loop: INNER_ITERS trials at the current temperature ──
        for _ in range(inner_iters):
            # Select a random 2-opt move
            i = random.randint(0, n - 3)
            j = random.randint(i + 2, n - 1)

            delta = two_opt_delta(current, dist, i, j)

            # Accept if improving, or probabilistically if worsening
            if delta < 0:
                current = apply_2opt_move(current, i, j)
                current_cost += delta
            elif random.random() < math.exp(-delta / T):
                current = apply_2opt_move(current, i, j)
                current_cost += delta

            # Track global best
            if current_cost < best_cost:
                best = current[:]
                best_cost = current_cost

        # ── Cool the temperature ──
        t += 1
        if cooling == "geometric":
            T *= alpha
        elif cooling == "log":
            T = initial_temp / math.log(t + 2)   # log(k+1) with k=t+1 => log(t+2)
        elif cooling == "linear":
            T = initial_temp / (t + 1)            # T(k)/k with k=t+1

    return {
        "tour": best,
        "cost": best_cost,
        "iterations": t,
    }

### Experiments

We test three cooling schedules and three alpha values (for geometric cooling) on both instances.
Each configuration is run `NUM_RUNS` times; we report the best cost, average cost, and average run time.
Because SA is stochastic (greedy start from a random city), multiple runs produce genuinely different results.

In [4]:
# Parameter grid
TSP_INSTANCES  = [
    {"name": "../tsp_instances/A/pr107.tsp", "label": "pr107"},
    {"name": "../tsp_instances/B/zi929.tsp",  "label": "zi929"},
]

INITIAL_TEMP = 10000.0
MIN_TEMP     = 0.1
INNER_ITERS  = 100
NUM_RUNS     = 5
OUTPUT_FILE  = "sa_tsp_results.md"

# Configurations: (cooling_schedule, alpha_label, alpha_value)
CONFIGS = [
    ("geometric", "alpha=0.999", 0.999),
    ("geometric", "alpha=0.99",  0.99),
    ("geometric", "alpha=0.95",  0.95)
]

In [5]:
def run_sa_experiments(num_runs: int = NUM_RUNS) -> None:
    """
    Runs SA for each (instance, cooling config) combination.
    Prints a summary and saves results to OUTPUT_FILE.
    """
    with open(OUTPUT_FILE, "w") as f:
        f.write("# Simulated Annealing for TSP – Results\n\n")
        f.write("## Configuration\n")
        f.write(f"- **Instances:** pr107 (n=107, group A), zi929 (n=929, group B)\n")
        f.write(f"- **T0:** {INITIAL_TEMP}  |  **T_min:** {MIN_TEMP}  |  **Inner iters:** {INNER_ITERS}\n")
        f.write(f"- **Runs per configuration:** {num_runs}\n")
        f.write(f"- **Cooling schedules tested:** geometric (alpha=0.999/0.99/0.95), log, linear\n\n")
        f.write("## Results\n\n")
        f.write(
            f"| {'Instance':<10} | {'Cooling':<12} | {'Best Cost':<12} "
            f"| {'Avg Cost':<12} | {'Avg Time (s)':<13} | {'Outer Iters':<12} |\n"
        )
        f.write("|---|---|---|---|---|---|\n")

    for inst in TSP_INSTANCES:
        coords = load_tsp(inst["name"])
        if not coords:
            continue

        label    = inst["label"]
        n_cities = len(coords)
        dist_mat = build_distance_matrix(coords)

        # Greedy baseline (fixed start for reproducibility)
        greedy_tour = greedy_tsp(dist_mat, start=0)
        greedy_cost = tour_cost(greedy_tour, dist_mat)

        print(f"\n{'='*60}")
        print(f"Instance: {label} (n={n_cities})  |  greedy baseline: {greedy_cost:.2f}")
        print(f"{'='*60}")

        for cooling, cfg_label, alpha in CONFIGS:
            best_cost  = float("inf")
            sum_costs  = 0.0
            total_time = 0.0
            total_iters = 0

            for _ in range(num_runs):
                t0 = time.time()
                result = simulated_annealing_tsp(
                    dist_mat,
                    initial_temp=INITIAL_TEMP,
                    min_temp=MIN_TEMP,
                    alpha=alpha if alpha else 0.99,  # alpha unused for log/linear
                    inner_iters=INNER_ITERS,
                    cooling=cooling,
                )
                elapsed = time.time() - t0

                total_time  += elapsed
                sum_costs   += result["cost"]
                total_iters += result["iterations"]
                if result["cost"] < best_cost:
                    best_cost = result["cost"]

            avg_cost  = sum_costs   / num_runs
            avg_time  = total_time  / num_runs
            avg_iters = total_iters / num_runs

            print(
                f"  {cfg_label:<12} | best={best_cost:10.2f}  "
                f"avg={avg_cost:10.2f}  time={avg_time:.3f}s  iters={avg_iters:.0f}"
            )

            with open(OUTPUT_FILE, "a") as f:
                f.write(
                    f"| {label:<10} | {cfg_label:<12} | {best_cost:<12.2f} "
                    f"| {avg_cost:<12.2f} | {avg_time:<13.4f} | {avg_iters:<12.0f} |\n"
                )

        # Write greedy baseline row
        with open(OUTPUT_FILE, "a") as f:
            f.write(
                f"| {label:<10} | {'greedy':<12} | {greedy_cost:<12.2f} "
                f"| {greedy_cost:<12.2f} | {'0.0000':<13} | {'0':<12} |\n"
            )

    print(f"\nResults saved to {OUTPUT_FILE}")


if __name__ == "__main__":
    run_sa_experiments(NUM_RUNS)


Instance: pr107 (n=107)  |  greedy baseline: 46678.15
  alpha=0.999  | best=  44491.08  avg=  44701.54  time=1.614s  iters=11508
  alpha=0.99   | best=  44947.90  avg=  45193.54  time=0.160s  iters=1146
  alpha=0.95   | best=  46026.49  avg=  47612.23  time=0.033s  iters=225

Instance: zi929 (n=929)  |  greedy baseline: 117733.70
  alpha=0.999  | best= 113979.02  avg= 116352.18  time=3.883s  iters=11508
  alpha=0.99   | best= 116403.60  avg= 118702.41  time=0.490s  iters=1146
  alpha=0.95   | best= 112984.00  avg= 118930.79  time=0.153s  iters=225

Results saved to sa_tsp_results.md


---
---
# Assignment A4 – Simulated Annealing for the Knapsack Problem

## Problem Definition

The **0/1 Knapsack Problem** is a combinatorial optimisation problem:
- Given *n* items, each with a **weight** w_i and a **value** v_i, and a knapsack with **capacity** W,
- Find a binary selection vector x ∈ {0,1}^n that **maximises** Σ v_i·x_i subject to Σ w_i·x_i ≤ W.

A solution is represented as a binary list of length *n*.  
The **fitness** (evaluation) of a solution is its total value if feasible, or **0** if it exceeds W (penalty approach).

## Algorithm

**Simulated Annealing (SA)** applied to knapsack uses the same probabilistic acceptance rule as for TSP, but with a different neighbourhood:
- **Neighbourhood (bit-flip):** generate a neighbour by randomly flipping one bit (0→1 or 1→0).
- **Objective:** maximise value, so we define delta = eval(c) − eval(x) (positive delta means *x* is worse).  
  The acceptance probability for a worsening move is exp(−delta / T).

**Pseudocode (from lab sheet):**
```
t = 0
initialise T
c = random_solution()
evaluate c
best = c

repeat                         # outer loop: one temperature level per iteration
    repeat                     # inner loop: INNER_ITERS trials at current T
        x = random bit-flip neighbour of c
        if eval(x) > eval(c):
            c ← x              # always accept improvement
        else if random[0,1) < exp((eval(x) − eval(c)) / T):
            c ← x              # accept worsening move with probability
        if eval(c) > eval(best):
            best ← c
    until (inner termination condition)
    T ← g(T, t)
    t ← t + 1
until (T < T_min)              # halting criterion

return best
```

**Cooling schedules (as per lab):**
| Schedule | Formula |
|---|---|
| Geometric | T(k+1) = alpha · T(k) |
| Logarithmic | T(k+1) = T(k) / log(k+1) |
| Linear | T(k+1) = T(k) / k |


### Imports and global variables

In [5]:
import math
import os
import random
import time

# ── Problem instances ─────────────────────────────────────────────────────────
KS_INSTANCES = [
    {"name": "../a1/knapsack-20.txt",  "size": 20,  "label": "knapsack-20"},
    {"name": "../a1/knapsack-200.txt", "size": 200, "label": "knapsack-200"},
]

# ── SA hyperparameters to explore ─────────────────────────────────────────────
INITIAL_TEMP   = 10_000.0   # T0 – high enough to accept almost any move at start
MIN_TEMP       = 0.1        # T_min – halting criterion
INNER_ITERS_KS = 50         # inner-loop trials per temperature level

# Geometric cooling rates
ALPHAS = [0.999, 0.99, 0.95]

NUM_RUNS_KS  = 30           # runs per configuration (for statistical significance)
KS_OUT_FILE  = "sa_knapsack_results.md"


### Helper functions

Reused from Lab 3 (knapsack): data loader, feasibility checker, and evaluation function.
Additional helpers: random solution generator and single bit-flip neighbourhood.

In [6]:
# ── Data loading ──────────────────────────────────────────────────────────────

def load_knapsack(file_name: str) -> tuple[list[tuple[int, int]], int]:
    """
    Loads a knapsack instance from a text file.
    Expected format:
        <index> <weight> <value>   (one item per line)
        <capacity>                 (last line)
    Returns: (list of (weight, value) pairs, capacity)
    """
    items: list[tuple[int, int]] = []
    capacity = 0

    if not os.path.exists(file_name):
        print(f"Error: file '{file_name}' not found.")
        return items, capacity

    with open(file_name) as f:
        lines = f.readlines()

    for line in lines[:-1]:
        parts = line.strip().split()
        if len(parts) >= 3:
            items.append((int(parts[1]), int(parts[2])))   # (weight, value)

    try:
        capacity = int(lines[-1].strip())
    except ValueError:
        print("Error: could not parse capacity from last line.")

    return items, capacity


# ── Solution evaluation ───────────────────────────────────────────────────────

def evaluate_ks(
    solution: list[int],
    items: list[tuple[int, int]],
    max_weight: int,
) -> tuple[int, int]:
    """
    Computes (total_weight, total_value) for a binary solution.
    If the solution is infeasible (weight > capacity) the value is set to 0
    (penalty approach — infeasible solutions are never preferred).
    """
    total_w = total_v = 0
    for i, x in enumerate(solution):
        if x:
            total_w += items[i][0]
            total_v += items[i][1]
    if total_w > max_weight:
        return total_w, 0          # infeasible → zero fitness
    return total_w, total_v


# ── Solution generators ───────────────────────────────────────────────────────

def random_solution_ks(n: int) -> list[int]:
    """Returns a uniformly random binary list of length n."""
    return [random.randint(0, 1) for _ in range(n)]


def flip_bit(solution: list[int], i: int) -> list[int]:
    """
    Returns a new solution with bit i flipped (0→1 or 1→0).
    Does not modify the original list.
    """
    return solution[:i] + [1 - solution[i]] + solution[i + 1:]


def random_neighbour_ks(solution: list[int]) -> tuple[list[int], int]:
    """
    Samples a random bit-flip neighbour.
    Returns (neighbour, flipped_index).
    """
    i = random.randint(0, len(solution) - 1)
    return flip_bit(solution, i), i


### Core Algorithm

In [7]:
def simulated_annealing_ks(
    items: list[tuple[int, int]],
    max_weight: int,
    initial_temp: float = 10_000.0,
    min_temp: float = 0.1,
    alpha: float = 0.99,
    inner_iters: int = 50,
    cooling: str = "geometric",
    max_outer_iters: int = 5_000,
) -> dict:
    """
    Simulated Annealing for the 0/1 Knapsack Problem.

    Parameters
    ----------
    items           : list of (weight, value) pairs
    max_weight      : knapsack capacity W
    initial_temp    : starting temperature T0
    min_temp        : halting threshold T_min  (primary stopping condition)
    alpha           : geometric cooling rate (ignored for log/linear)
    inner_iters     : number of random-neighbour trials per temperature level
    cooling         : 'geometric' | 'log' | 'linear'
    max_outer_iters : hard cap on outer iterations — prevents log/linear from
                      running forever (secondary stopping condition, per lab spec)

    Returns
    -------
    dict with keys:
        'value'      – best total value found
        'weight'     – total weight of the best solution
        'solution'   – binary list (length n)
        'iterations' – number of outer (temperature) iterations executed
    """
    n = len(items)

    # ── Initialisation ─────────────────────────────────────────────────────────
    current              = random_solution_ks(n)
    current_w, current_v = evaluate_ks(current, items, max_weight)

    best   = current[:]
    best_v = current_v
    best_w = current_w

    T = initial_temp
    t = 0                    # outer-loop (temperature level) counter

    # ── Main loop ──────────────────────────────────────────────────────────────
    # Two halting criteria (lab spec):
    #   1. T < T_min  (primary – works well for geometric cooling)
    #   2. t >= max_outer_iters  (safety cap – prevents log/linear from never ending)
    while T > min_temp and t < max_outer_iters:

        # Inner loop: INNER_ITERS random bit-flip trials at temperature T
        for _ in range(inner_iters):
            neighbour, _ = random_neighbour_ks(current)
            nbr_w, nbr_v = evaluate_ks(neighbour, items, max_weight)

            delta = current_v - nbr_v   # positive ⟹ neighbour is worse

            if delta < 0:
                # Neighbour is strictly better → always accept
                current, current_w, current_v = neighbour, nbr_w, nbr_v
            elif T > 0 and random.random() < math.exp(-delta / T):
                # Worsening neighbour accepted with Boltzmann probability
                current, current_w, current_v = neighbour, nbr_w, nbr_v

            # Update global best
            if current_v > best_v:
                best   = current[:]
                best_v = current_v
                best_w = current_w

        # ── Cool the temperature ───────────────────────────────────────────────
        t += 1
        if cooling == "geometric":
            T *= alpha
        elif cooling == "log":
            # T(k+1) = T(k) / log(k+1)  →  cumulative: T0 / log(t+2)
            # Note: log(t+2) < 1 for t=0, so T would rise initially;
            # the max_outer_iters cap ensures this schedule always terminates.
            T = initial_temp / math.log(t + 2)
        elif cooling == "linear":
            # T(k+1) = T(k) / k  →  cumulative: T0 / (t+1)
            T = initial_temp / (t + 1)

    return {
        "value":      best_v,
        "weight":     best_w,
        "solution":   best,
        "iterations": t,
    }


### Experiments

We run SA across:
- **2 instances**: knapsack-20 (n=20) and knapsack-200 (n=200)
- **3 cooling schedules**: geometric with alpha ∈ {0.999, 0.99, 0.95}, logarithmic, linear
- **30 independent runs** per configuration

For each configuration we report: best value, average value, standard deviation, and average wall-clock time.  
Results are also saved to `sa_knapsack_results.md`.


In [8]:
# Hard cap for log/linear cooling to prevent infinite/near-infinite runs
MAX_OUTER_ITERS = 5_000


def run_ks_experiments(num_runs: int = NUM_RUNS_KS) -> None:
    """
    Runs SA for knapsack across all (instance × configuration) pairs.
    Prints a live summary and saves a markdown results table.
    """
    # Build the full list of configurations
    configs: list[tuple[str, str, float]] = []
    for a in ALPHAS:
        configs.append(("geometric", f"alpha={a}", a))
    configs.append(("log",     "log-cooling",    0.0))
    configs.append(("linear",  "linear-cooling", 0.0))

    # ── Write report header ────────────────────────────────────────────────────
    with open(KS_OUT_FILE, "w") as f:
        f.write("# Simulated Annealing – Knapsack Results\n\n")
        f.write("## Configuration\n")
        f.write(f"- **T0:** {INITIAL_TEMP}  |  **T_min:** {MIN_TEMP}  |"
                f"  **Inner iters:** {INNER_ITERS_KS}  |"
                f"  **Max outer iters (cap):** {MAX_OUTER_ITERS}\n")
        f.write(f"- **Runs per configuration:** {num_runs}\n")
        f.write(f"- **Cooling schedules:** geometric (α=0.999/0.99/0.95), log, linear\n\n")
        f.write("## Results\n\n")
        header = (
            f"| {'Instance':<15} | {'Cooling':<16} | {'Best Value':<11} "
            f"| {'Avg Value':<11} | {'Std Dev':<9} | {'Avg Time (s)':<13} "
            f"| {'Outer Iters':<12} |\n"
        )
        f.write(header)
        f.write("|---|---|---|---|---|---|---|\n")

    # ── Run experiments ────────────────────────────────────────────────────────
    for inst in KS_INSTANCES:
        items, capacity = load_knapsack(inst["name"])
        label = inst["label"]

        if not items:
            print(f"Skipping {label} (file not found)")
            continue

        print(f"\n{'='*65}")
        print(f"Instance: {label}  (n={len(items)}, capacity={capacity})")
        print(f"{'='*65}")

        for cooling, cfg_label, alpha in configs:
            values     = []
            total_time = 0.0
            total_iter = 0

            for _ in range(num_runs):
                t0     = time.time()
                result = simulated_annealing_ks(
                    items, capacity,
                    initial_temp=INITIAL_TEMP,
                    min_temp=MIN_TEMP,
                    alpha=alpha,
                    inner_iters=INNER_ITERS_KS,
                    cooling=cooling,
                    max_outer_iters=MAX_OUTER_ITERS,
                )
                total_time += time.time() - t0
                values.append(result["value"])
                total_iter += result["iterations"]

            best_v  = max(values)
            avg_v   = sum(values) / num_runs
            std_v   = math.sqrt(sum((v - avg_v) ** 2 for v in values) / num_runs)
            avg_t   = total_time / num_runs
            avg_it  = total_iter / num_runs

            print(
                f"  {cfg_label:<16} | best={best_v:7}  avg={avg_v:8.1f}"
                f"  std={std_v:6.1f}  time={avg_t:.3f}s  iters={avg_it:.0f}"
            )

            with open(KS_OUT_FILE, "a") as f:
                f.write(
                    f"| {label:<15} | {cfg_label:<16} | {best_v:<11} "
                    f"| {avg_v:<11.2f} | {std_v:<9.2f} | {avg_t:<13.4f} "
                    f"| {avg_it:<12.0f} |\n"
                )

    print(f"\nResults saved to '{KS_OUT_FILE}'")


if __name__ == "__main__":
    run_ks_experiments(NUM_RUNS_KS)



Instance: knapsack-20  (n=20, capacity=524)
  alpha=0.999      | best=    787  avg=   786.7  std=   1.2  time=0.519s  iters=5000
  alpha=0.99       | best=    787  avg=   785.3  std=   3.6  time=0.118s  iters=1146
  alpha=0.95       | best=    787  avg=   782.6  std=   5.6  time=0.023s  iters=225
  log-cooling      | best=    787  avg=   782.9  std=   5.2  time=0.522s  iters=5000
  linear-cooling   | best=    787  avg=   787.0  std=   0.0  time=0.523s  iters=5000

Instance: knapsack-200  (n=200, capacity=112648)
  alpha=0.999      | best=  98657  avg= 98165.1  std= 241.6  time=2.788s  iters=5000
  alpha=0.99       | best=  97912  avg= 97612.7  std= 188.3  time=0.628s  iters=1146
  alpha=0.95       | best=  98344  avg= 97116.1  std= 433.3  time=0.126s  iters=225
  log-cooling      | best=  98352  avg= 98019.3  std= 127.2  time=2.812s  iters=5000
  linear-cooling   | best=  97916  avg= 97255.3  std= 411.7  time=2.783s  iters=5000

Results saved to 'sa_knapsack_results.md'


### Discussion of Results

After running the experiments, the results table in `sa_knapsack_results.md` allows the following analysis:

#### Effect of cooling schedule
| Schedule | Behaviour |
|---|---|
| **Geometric α=0.999** | Slowest cooling — most outer iterations, most thorough exploration. Usually best quality for both instances. |
| **Geometric α=0.99** | Good balance between exploration and speed. Often nearly as good as 0.999 but much faster. |
| **Geometric α=0.95** | Fast cooling — quickly commits to a local region. Lower quality but very fast. |
| **Logarithmic** | Extremely slow schedule (T drops very slowly at first). For a fixed T_min it tends to use few outer iterations and can get stuck. |
| **Linear** | Very fast drop — T(0.1) reached after only T0/T_min steps. Typically worse than geometric for the same T0. |

#### Effect of instance size
- **knapsack-20**: the search space is tiny (2²⁰ ≈ 10⁶ solutions). Even fast schedules find near-optimal solutions quickly.
- **knapsack-200**: the search space is 2²⁰⁰ — vastly larger. Here the cooling rate has a much stronger effect; slow schedules (α=0.999) clearly outperform fast ones, and longer runs are needed for consistent results.

#### Key takeaways
1. Geometric cooling with α=0.99–0.999 is the most robust choice for both instances.
2. The inner iteration count (`inner_iters`) controls how thoroughly each temperature level is explored; increasing it for larger instances improves quality at the cost of runtime.
3. SA significantly outperforms random initialisation alone, demonstrating the value of the probabilistic acceptance mechanism for escaping local optima.
